In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os

ROOT = '/content/drive/MyDrive/amsdds_v2'
for d in ['src', 'configs', 'notebooks', 'cache', 'outputs', 'weights']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

print(os.path.isdir(ROOT), os.listdir(ROOT))

True ['src', 'configs', 'notebooks', 'cache', 'outputs', 'weights']


In [ ]:
!pip install -q gdown
import gdown, os, torch

CKPT = f'{ROOT}/weights/panderm_base.pth'
if not os.path.isfile(CKPT):
    gdown.download(id='17J4MjsZu3gdBP6xAQi_NMDVvH65a00HB', output=CKPT, quiet=False)
print(f'{os.path.getsize(CKPT)/1e6:.0f} MB')

# INSPECT BEFORE TRUSTING. My adapter assumes plain-timm ViT key names.
obj = torch.load(CKPT, map_location='cpu', weights_only=False)
if isinstance(obj, dict):
    for k in ('model', 'state_dict', 'module'):
        if k in obj:
            print(f'wrapper key: {k}'); obj = obj[k]; break
keys = list(obj.keys())
print(f'{len(keys)} tensors\nfirst 15:'); [print(' ', k) for k in keys[:15]]
print('has blocks.0.attn.qkv.weight:', 'blocks.0.attn.qkv.weight' in keys)
print('has pos_embed:', any('pos_embed' in k for k in keys))

Downloading...
From (original): https://drive.google.com/uc?id=17J4MjsZu3gdBP6xAQi_NMDVvH65a00HB
From (redirected): https://drive.google.com/uc?id=17J4MjsZu3gdBP6xAQi_NMDVvH65a00HB&confirm=t&uuid=15ff326e-4080-46fd-92f9-d6e582561744
To: /content/drive/MyDrive/amsdds_v2/weights/panderm_base.pth
100%|██████████| 343M/343M [00:07<00:00, 44.7MB/s]


343 MB
186 tensors
first 15:
  cls_token
  pos_embed
  patch_embed.proj.weight
  patch_embed.proj.bias
  blocks.0.gamma_1
  blocks.0.gamma_2
  blocks.0.norm1.weight
  blocks.0.norm1.bias
  blocks.0.attn.q_bias
  blocks.0.attn.v_bias
  blocks.0.attn.qkv.weight
  blocks.0.attn.proj.weight
  blocks.0.attn.proj.bias
  blocks.0.norm2.weight
  blocks.0.norm2.bias
has blocks.0.attn.qkv.weight: True
has pos_embed: True


In [4]:
# =============================================================================
# CELL 0 — put the uploaded files where the imports expect them
# =============================================================================
# You've mounted and made the folders. This finds the four files wherever they
# landed and moves them into src/ and configs/. Safe to re-run.
import os, glob, shutil, sys

ROOT = '/content/drive/MyDrive/amsdds_v2'
WANT = {'backbones.py': 'src', 'data.py': 'src', 'heads.py': 'src',
        'splits.csv': 'configs'}

for fname, dest in WANT.items():
    target = f'{ROOT}/{dest}/{fname}'
    if os.path.isfile(target):
        print(f'ok       {dest}/{fname}'); continue
    hits = [p for p in glob.glob(f'{ROOT}/**/{fname}', recursive=True)
            if os.path.isfile(p)]
    if hits:
        shutil.move(hits[0], target)
        print(f'moved    {hits[0]} -> {dest}/{fname}')
    else:
        print(f'MISSING  {fname}  <-- upload it')

print('\nsrc/    ', sorted(os.listdir(f'{ROOT}/src')))
print('configs/', sorted(os.listdir(f'{ROOT}/configs')))



ok       src/backbones.py
ok       src/data.py
ok       src/heads.py
ok       configs/splits.csv

src/     ['__pycache__', 'backbones.py', 'data.py', 'finetune.py', 'heads.py']
configs/ ['splits.csv']


In [5]:

# =============================================================================
# CELL 1 — install, path, GPU check
# =============================================================================
import subprocess
subprocess.run('pip install -q timm==1.0.* kagglehub', shell=True, check=True)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'torch {torch.__version__} | {gpu}')
if 'T4' in gpu:
    print('T4: fp16 autocast is correct (no bf16 on Turing). Batch 64 @224 fits.')



torch 2.11.0+cu128 | Tesla T4
T4: fp16 autocast is correct (no bf16 on Turing). Batch 64 @224 fits.


In [6]:

# =============================================================================
# CELL 2 — HAM10000 + manifest against YOUR split
# =============================================================================
import kagglehub, numpy as np
DATA = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
print(DATA)

from src.data import build_manifest, class_prior, CLASSES, MALIGNANT

df = build_manifest(f'{ROOT}/configs/splits.csv', [DATA])
prior = class_prior(df, 'train')
print('\nprior:', dict(zip(CLASSES, prior.round(4))))

# No lesion may span two splits. If this fires, test metrics are inflated and
# nothing downstream means anything.
if 'lesion_id' in df.columns:
    leak = df.groupby('lesion_id')['split'].nunique()
    assert (leak == 1).all(), f'LEAK: {(leak > 1).sum()} lesions span splits'
    print('no lesion leakage')



Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
/kaggle/input/skin-cancer-mnist-ham10000
[data] 10015 images
dx     akiec  bcc  bkl  df  mel    nv  vasc
split                                      
test      34   89  168   9  146   974    21
train    230  358  772  89  782  4796    89
val       63   67  159  17  185   935    32

prior: {'akiec': np.float32(0.0323), 'bcc': np.float32(0.0503), 'bkl': np.float32(0.1085), 'df': np.float32(0.0125), 'mel': np.float32(0.1099), 'nv': np.float32(0.674), 'vasc': np.float32(0.0125)}
no lesion leakage


In [ ]:

# =============================================================================
# CELL 3 — backbone + the assert that saves you an hour
# =============================================================================
from src.backbones import build_backbone
from PIL import Image

PANDERM_CKPT = f'{ROOT}/weights/panderm_base.pth'
BACKBONE     = 'panderm_base'      # -> 'dinov2_base' if the weights don't land

try:
    bb = build_backbone(BACKBONE, 'cuda', panderm_ckpt=PANDERM_CKPT)
except Exception as e:
    print(f'!! {type(e).__name__}: {e}\n!! falling back to dinov2_base')
    BACKBONE = 'dinov2_base'
    bb = build_backbone(BACKBONE, 'cuda')

probe = torch.stack([bb.transform(Image.new('RGB', (400, 300), (c, 120, 100)))
                     for c in (60, 120, 180, 240)])
f = bb.encode(probe)
d = torch.cdist(f, f)
print(f'{bb.name}  dim={bb.dim}  img={bb.img_size}  '
      f'std={f.std():.4f}  min_pairwise={d[d>0].min():.4f}')
assert f.std() > 1e-3 and d[d > 0].min() > 1e-3, 'backbone did not load'
print('backbone live')



[panderm] loaded 186 tensors | missing=0 unexpected=0
[panderm] clean load, all weights mapped
panderm_base  dim=768  img=224  std=0.5047  min_pairwise=5.9019
backbone live


In [ ]:
import re
p = f'{ROOT}/src/backbones.py'
s = open(p).read()
old = "             use_rel_pos_bias=False, use_shared_rel_pos_bias=False,\n             num_classes=0)"
new = "             use_rel_pos_bias=False, use_shared_rel_pos_bias=False,\n             global_pool='token', num_classes=0)"
assert old in s, 'pattern not found — check the file uploaded correctly'
open(p, 'w').write(s.replace(old, new))
print(open(p).read().count("global_pool='token'"), "<- must be 1")

1 <- must be 1


In [ ]:

# =============================================================================
# CELL 4 — throughput probe BEFORE committing to the full run
# =============================================================================
# Measures real cost including JPEG decode + colour constancy, which on
# Colab's 2 vCPUs is usually the bottleneck rather than the T4.
from src.data import LesionDS
from torch.utils.data import DataLoader
import time

VIEWS, BATCH, WORKERS = 4, 64, 2

_probe = df[df.split == 'train'].head(256).reset_index(drop=True)
_dl = DataLoader(LesionDS(_probe, bb.transform, views=VIEWS),
                 batch_size=BATCH, num_workers=WORKERS)
t0 = time.perf_counter()
for x, _ in _dl:
    bb.encode(x.reshape(-1, *x.shape[-3:]))
per = (time.perf_counter() - t0) / len(_probe)
print(f'{per*1e3:.0f} ms/img -> all {len(df)} images in ~{per*len(df)/60:.0f} min')
print('Over ~40 min? Set VIEWS=1 (halves it) or BACKBONE="dinov2_small".')



39 ms/img -> all 10015 images in ~7 min
Over ~40 min? Set VIEWS=1 (halves it) or BACKBONE="dinov2_small".


In [ ]:
print(BACKBONE)

panderm_base


In [ ]:

# =============================================================================
# CELL 5 — extract once, cache to Drive
# =============================================================================
def extract(split):
    out = f'{ROOT}/cache/{BACKBONE}_{split}_v{VIEWS}.npz'
    if os.path.exists(out):
        print(f'{split}: cached'); return out
    sub = df[df.split == split].reset_index(drop=True)
    dl = DataLoader(LesionDS(sub, bb.transform, views=VIEWS), batch_size=BATCH,
                    num_workers=WORKERS, shuffle=False, pin_memory=True)
    feats, t0 = [], time.perf_counter()
    for k, (x, _) in enumerate(dl):
        n, v = (x.shape[0], 1) if x.ndim == 4 else (x.shape[0], x.shape[1])
        fx = bb.encode(x.reshape(-1, *x.shape[-3:])).reshape(n, v, -1)
        feats.append(fx.numpy().astype(np.float32))
        if k % 20 == 0:
            print(f'  {split} {k*BATCH}/{len(sub)}  {time.perf_counter()-t0:.0f}s', flush=True)
    np.savez_compressed(out, feats=np.concatenate(feats),
                        y=sub.y.to_numpy(), image_id=sub.image_id.to_numpy())
    print(f'{split}: {(time.perf_counter()-t0)/60:.1f} min -> {out}')
    return out

paths = {s: extract(s) for s in ['train', 'val', 'test']}
Z  = {s: np.load(p, allow_pickle=True) for s, p in paths.items()}
F_ = {s: Z[s]['feats'] for s in Z}      # [N, VIEWS, dim]
Y_ = {s: Z[s]['y']     for s in Z}
print({s: F_[s].shape for s in F_})



  train 0/7116  7s
  train 1280/7116  57s
  train 2560/7116  104s
  train 3840/7116  151s
  train 5120/7116  198s
  train 6400/7116  244s
train: 4.5 min -> /content/drive/MyDrive/amsdds_v2/cache/panderm_base_train_v4.npz
  val 0/1458  7s
  val 1280/1458  53s
val: 0.9 min -> /content/drive/MyDrive/amsdds_v2/cache/panderm_base_val_v4.npz
  test 0/1441  5s
  test 1280/1441  53s
test: 0.9 min -> /content/drive/MyDrive/amsdds_v2/cache/panderm_base_test_v4.npz
{'train': (7116, 4, 768), 'val': (1458, 4, 768), 'test': (1441, 4, 768)}


In [ ]:

# =============================================================================
# CELL 6 — train the head (seconds — sweep freely from here)
# =============================================================================
from src.heads import (standardize, train_head, logits_of, fit_temperature,
                       evaluate, sweep_risk_threshold)
import torch.nn.functional as Fn

Xtr = F_['train'][:, 0]                 # canonical view only for training
mu, sd = standardize(Xtr)
S = lambda a: (a - mu) / sd

head, hist = train_head(S(Xtr), Y_['train'], S(F_['val'][:, 0]), Y_['val'],
                        prior, dim=bb.dim, hidden=512, drop=0.3,
                        tau=1.0, smooth=0.1, mixup=0.2, epochs=60)

T = fit_temperature(logits_of(head, S(F_['val'][:, 0])), Y_['val'])
print(f'\ntemperature T = {T:.3f}   (v1 was 0.959 — below 1.0 SHARPENS)')

def probs(split, view=0):
    lg = logits_of(head, S(F_[split][:, view]))
    return Fn.softmax(torch.tensor(lg) / T, 1).numpy()

p_test = probs('test')
m = evaluate(p_test, Y_['test'])
print(f"\n{'':22}{'new':>9}{'v1 L2':>9}")
print(f"{'accuracy':22}{m['acc']:>9.4f}{0.857:>9.4f}")
print(f"{'macro F1':22}{m['macro_f1']:>9.4f}{0.736:>9.4f}")
print(f"{'ECE':22}{m['ece']:>9.4f}{'—':>9}")
print(f"{'mal sensitivity':22}{m['mal_sensitivity']:>9.4f}{0.966:>9.4f}")
print(f"{'mal specificity':22}{m['mal_specificity']:>9.4f}{0.484:>9.4f}")
print('\nper-class F1:', m['per_class_f1'])

best, _ = sweep_risk_threshold(p_test, Y_['test'], target_sens=0.966)
if best:
    print(f'\nat sens>=0.966: thr {best[0]:.3f} -> specificity {best[2]:.3f} (v1 0.484)')



/content/drive/MyDrive/amsdds_v2/src/heads.py:83: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  tot += float(loss) * len(xb)


  ep  0  loss 1.0272  val_acc 0.3580  val_f1 0.3939
  ep 10  loss 0.7282  val_acc 0.4499  val_f1 0.4388
  ep 20  loss 0.6340  val_acc 0.5192  val_f1 0.4793
  ep 30  loss 0.5999  val_acc 0.5034  val_f1 0.4693
  ep 40  loss 0.5779  val_acc 0.4966  val_f1 0.4635
  ep 50  loss 0.5688  val_acc 0.5014  val_f1 0.4707
  ep 59  loss 0.6150  val_acc 0.4883  val_f1 0.4655
  best epoch 9 val_f1 0.4917

temperature T = 0.827   (v1 was 0.959 — below 1.0 SHARPENS)

                            new    v1 L2
accuracy                 0.5142   0.8570
macro F1                 0.4167   0.7360
ECE                      0.0746        —
mal sensitivity          0.8773   0.9660
mal specificity          0.7474   0.4840

per-class F1: {'akiec': 0.3279, 'bcc': 0.6396, 'bkl': 0.641, 'df': 0.0611, 'mel': 0.4636, 'nv': 0.6359, 'vasc': 0.1481}

at sens>=0.966: thr 0.140 -> specificity 0.427 (v1 0.484)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

Xtr, ytr = S(F_['train'][:, 0]), Y_['train']
Xte, yte = S(F_['test'][:, 0]),  Y_['test']

lr = LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1).fit(Xtr, ytr)
pred = lr.predict(Xte)

print(f'plain linear probe   acc {accuracy_score(yte, pred):.4f}   '
      f'macroF1 {f1_score(yte, pred, average="macro"):.4f}')
print(f'majority-class floor acc {(yte == 5).mean():.4f}')
print('per-class F1:', dict(zip(CLASSES,
      f1_score(yte, pred, average=None, labels=range(7)).round(3))))

plain linear probe   acc 0.8015   macroF1 0.6392
majority-class floor acc 0.6759
per-class F1: {'akiec': np.float64(0.444), 'bcc': np.float64(0.694), 'bkl': np.float64(0.611), 'df': np.float64(0.5), 'mel': np.float64(0.508), 'nv': np.float64(0.906), 'vasc': np.float64(0.811)}


In [ ]:
import itertools, pandas as pd
from src.heads import train_head, logits_of, fit_temperature, evaluate
import torch.nn.functional as Fn

Xtr_c, Xva_c = S(F_['train'][:, 0]), S(F_['val'][:, 0])
Xtr_m, Xva_m = S(F_['train'].mean(1)), S(F_['val'].mean(1))   # TTA-averaged

rows = []
for feats, tag in [((Xtr_c, Xva_c), 'canon'), ((Xtr_m, Xva_m), 'ttaavg')]:
    for hidden, tau, mix, dr in itertools.product([0, 512], [0.0, 0.25, 0.5], [0.0, 0.2], [0.1]):
        h, _ = train_head(feats[0], Y_['train'], feats[1], Y_['val'], prior,
                          dim=bb.dim, hidden=hidden, drop=dr, tau=tau,
                          smooth=0.0 if mix == 0 else 0.1, mixup=mix,
                          epochs=40, verbose=False)
        lv = logits_of(h, feats[1]); t = fit_temperature(lv, Y_['val'])
        mm = evaluate(Fn.softmax(torch.tensor(lv) / t, 1).numpy(), Y_['val'])
        rows.append({'feat': tag, 'hidden': hidden, 'tau': tau, 'mix': mix,
                     'T': round(t, 2), 'val_acc': round(mm['acc'], 4),
                     'val_f1': round(mm['macro_f1'], 4),
                     'spec': round(mm['mal_specificity'], 4)})
        print(rows[-1], flush=True)

print(pd.DataFrame(rows).sort_values('val_f1', ascending=False).to_string(index=False))

{'feat': 'canon', 'hidden': 0, 'tau': 0.0, 'mix': 0.0, 'T': 0.87, 'val_acc': 0.7791, 'val_f1': 0.6233, 'spec': 0.7533}
{'feat': 'canon', 'hidden': 0, 'tau': 0.0, 'mix': 0.2, 'T': 0.62, 'val_acc': 0.7949, 'val_f1': 0.6385, 'spec': 0.7997}
{'feat': 'canon', 'hidden': 0, 'tau': 0.25, 'mix': 0.0, 'T': 0.77, 'val_acc': 0.7743, 'val_f1': 0.6212, 'spec': 0.7603}
{'feat': 'canon', 'hidden': 0, 'tau': 0.25, 'mix': 0.2, 'T': 0.53, 'val_acc': 0.7908, 'val_f1': 0.639, 'spec': 0.7935}
{'feat': 'canon', 'hidden': 0, 'tau': 0.5, 'mix': 0.0, 'T': 0.72, 'val_acc': 0.7654, 'val_f1': 0.612, 'spec': 0.7323}
{'feat': 'canon', 'hidden': 0, 'tau': 0.5, 'mix': 0.2, 'T': 0.52, 'val_acc': 0.762, 'val_f1': 0.6102, 'spec': 0.7524}
{'feat': 'canon', 'hidden': 512, 'tau': 0.0, 'mix': 0.0, 'T': 1.85, 'val_acc': 0.845, 'val_f1': 0.6995, 'spec': 0.8644}
{'feat': 'canon', 'hidden': 512, 'tau': 0.0, 'mix': 0.2, 'T': 0.77, 'val_acc': 0.8402, 'val_f1': 0.7024, 'spec': 0.8863}
{'feat': 'canon', 'hidden': 512, 'tau': 0.25, 

In [ ]:
# =============================================================================
# CELL F0 — save the frozen-probe result BEFORE risking anything
# =============================================================================
# ttaavg / hidden=512 / tau=0 / mixup=0.2 won the sweep on val. Lock the test
# number in now so you have a defensible result even if fine-tuning fails.
import torch, numpy as np, json
import torch.nn.functional as Fn
from src.heads import (standardize, train_head, logits_of, fit_temperature,
                       evaluate, sweep_risk_threshold)
from src.data import CLASSES

Xtr_m = F_['train'].mean(1); mu, sd = standardize(Xtr_m)
S = lambda a: (a - mu) / sd

probe, _ = train_head(S(Xtr_m), Y_['train'], S(F_['val'].mean(1)), Y_['val'],
                      prior, dim=bb.dim, hidden=512, drop=0.1, tau=0.0,
                      smooth=0.1, mixup=0.2, epochs=40)
Tp = fit_temperature(logits_of(probe, S(F_['val'].mean(1))), Y_['val'])
p_probe = Fn.softmax(torch.tensor(logits_of(probe, S(F_['test'].mean(1)))) / Tp, 1).numpy()
m_probe = evaluate(p_probe, Y_['test'])
print('FROZEN PROBE (test):', {k: round(v, 4) for k, v in m_probe.items()
                               if k != 'per_class_f1'})

torch.save({'head': probe.state_dict(), 'feat_mu': mu, 'feat_sd': sd,
            'hparams': {'hidden': 512, 'drop': 0.1}, 'temperature': Tp,
            'meta': {'backbone': 'panderm_base', 'mode': 'frozen_probe',
                     'dim': bb.dim, 'classes': CLASSES, 'test': m_probe}},
           f'{ROOT}/outputs/layer2_panderm_probe.pt')
print('saved fallback result')



  ep  0  loss 1.1546  val_acc 0.7888  val_f1 0.5612
  ep 10  loss 0.7221  val_acc 0.8457  val_f1 0.6828
  ep 20  loss 0.6239  val_acc 0.8361  val_f1 0.6733
  ep 30  loss 0.5922  val_acc 0.8422  val_f1 0.6870
  ep 39  loss 0.5873  val_acc 0.8450  val_f1 0.6913
  best epoch 5 val_f1 0.7167
FROZEN PROBE (test): {'acc': 0.8432, 'macro_f1': 0.6891, 'ece': 0.0226, 'mal_sensitivity': 0.7955, 'mal_specificity': 0.8925}
saved fallback result


In [ ]:
from src.heads import sweep_risk_threshold
best, rows = sweep_risk_threshold(p_probe, Y_['test'], target_sens=0.966)
print('probe @ sens>=0.966:', 'none reachable' if not best else
      f'thr {best[0]:.3f} -> spec {best[2]:.3f}   (v1: 0.484)')
for s in (0.90, 0.95, 0.966):
    b, _ = sweep_risk_threshold(p_probe, Y_['test'], target_sens=s)
    print(f'  sens>={s}: spec {b[2]:.3f}' if b else f'  sens>={s}: unreachable')

probe @ sens>=0.966: thr 0.040 -> spec 0.621   (v1: 0.484)
  sens>=0.9: spec 0.742
  sens>=0.95: spec 0.655
  sens>=0.966: spec 0.621


In [ ]:
import os
p = f'{ROOT}/src/finetune.py'
print(os.path.isfile(p), os.path.getsize(p) if os.path.isfile(p) else 0)
print(sorted(os.listdir(f'{ROOT}/src')))

True 10980
['__pycache__', 'backbones.py', 'data.py', 'finetune.py', 'heads.py']


In [ ]:
import shutil, importlib, sys, os
shutil.rmtree(f'{ROOT}/src/__pycache__', ignore_errors=True)
importlib.invalidate_caches()

for m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[m]

from src.finetune import (cache_cc_images, build_finetune_model, finetune,
                          predict_tta, logits_plain)
print('ok')

ok


In [7]:

# =============================================================================
# CELL F1 — colour-constancy cache to LOCAL disk (~4 min, run once)
# =============================================================================
# Do NOT skip. Without it every epoch re-runs shades-of-gray on CPU and the
# T4 sits idle waiting for 2 vCPUs.
from src.finetune import cache_cc_images

sub = {s: df[df.split == s].reset_index(drop=True) for s in ['train', 'val', 'test']}
cc = {s: cache_cc_images(sub[s]) for s in sub}
Yd = {s: sub[s].y.to_numpy() for s in sub}
print({s: len(cc[s]) for s in cc})



  0/7116  0s
  2000/7116  84s
  4000/7116  165s
  6000/7116  245s
cached 7116 in 4.8 min -> /content/cc_cache
  0/1458  0s
cached 1458 in 1.0 min -> /content/cc_cache
  0/1441  0s
cached 1441 in 1.0 min -> /content/cc_cache
{'train': 7116, 'val': 1458, 'test': 1441}


In [ ]:

# =============================================================================
# CELL F2 — build the model, check memory fits
# =============================================================================
from src.finetune import build_finetune_model, finetune, predict_tta, logits_plain

model = build_finetune_model(f'{ROOT}/weights/panderm_base.pth',
                             n_classes=7, drop_path=0.2)

torch.cuda.reset_peak_memory_stats()
_x = torch.randn(32, 3, 224, 224, device='cuda')
with torch.autocast('cuda', torch.float16):
    model(_x).sum().backward()
print(f'peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB at batch 32 (T4 has 16)')
model.zero_grad(set_to_none=True); del _x; torch.cuda.empty_cache()



[ft] PanDerm loaded, fresh 7-class head
peak 3.4 GB at batch 32 (T4 has 16)


In [ ]:

# =============================================================================
# CELL F3 — fine-tune (~60-90 min). Watch the first 3 epochs.
# =============================================================================
model, hist = finetune(model, cc['train'], Yd['train'], cc['val'], Yd['val'],
                       epochs=50, batch=128, accum=1, lr=5e-4, wd=0.05,
                       layer_decay=0.65, warmup=10, mixup_a=0.8, cutmix_a=1.0,
                       workers=2, out_path=f'{ROOT}/outputs/panderm_ft_best.pt')

[ft] 28 param groups, lr 1.85e-06 .. 5.00e-04
ep  0  loss 1.9247  val_acc 0.5487  val_f1 0.3372  65s  *
ep  1  loss 1.5932  val_acc 0.6392  val_f1 0.5133  69s  *
ep  2  loss 1.4302  val_acc 0.7284  val_f1 0.5731  70s  *
ep  3  loss 1.3827  val_acc 0.7167  val_f1 0.6012  74s  *
ep  4  loss 1.3004  val_acc 0.7675  val_f1 0.6835  74s  *
ep  5  loss 1.2512  val_acc 0.7284  val_f1 0.6911  74s  *
ep  6  loss 1.2692  val_acc 0.8073  val_f1 0.7399  74s  *
ep  7  loss 1.2163  val_acc 0.7689  val_f1 0.7069  72s
ep  8  loss 1.1534  val_acc 0.7812  val_f1 0.6993  72s
ep  9  loss 1.1954  val_acc 0.8018  val_f1 0.7219  74s
ep 10  loss 1.1540  val_acc 0.7798  val_f1 0.6769  73s
ep 11  loss 1.1650  val_acc 0.7812  val_f1 0.7382  72s
ep 12  loss 1.1443  val_acc 0.7826  val_f1 0.7352  72s
ep 13  loss 1.1588  val_acc 0.8457  val_f1 0.7561  78s  *
ep 14  loss 1.1633  val_acc 0.8320  val_f1 0.7549  74s
ep 15  loss 1.0661  val_acc 0.8313  val_f1 0.7526  73s
ep 16  loss 1.1386  val_acc 0.8477  val_f1 0.7451 

In [8]:
import re, numpy as np

LOG = """
ep  0  loss 1.9247  val_acc 0.5487  val_f1 0.3372  65s  *
ep  1  loss 1.5932  val_acc 0.6392  val_f1 0.5133  69s  *
ep  2  loss 1.4302  val_acc 0.7284  val_f1 0.5731  70s  *
ep  3  loss 1.3827  val_acc 0.7167  val_f1 0.6012  74s  *
ep  4  loss 1.3004  val_acc 0.7675  val_f1 0.6835  74s  *
ep  5  loss 1.2512  val_acc 0.7284  val_f1 0.6911  74s  *
ep  6  loss 1.2692  val_acc 0.8073  val_f1 0.7399  74s  *
ep  7  loss 1.2163  val_acc 0.7689  val_f1 0.7069  72s
ep  8  loss 1.1534  val_acc 0.7812  val_f1 0.6993  72s
ep  9  loss 1.1954  val_acc 0.8018  val_f1 0.7219  74s
ep 10  loss 1.1540  val_acc 0.7798  val_f1 0.6769  73s
ep 11  loss 1.1650  val_acc 0.7812  val_f1 0.7382  72s
ep 12  loss 1.1443  val_acc 0.7826  val_f1 0.7352  72s
ep 13  loss 1.1588  val_acc 0.8457  val_f1 0.7561  78s  *
ep 14  loss 1.1633  val_acc 0.8320  val_f1 0.7549  74s
ep 15  loss 1.0661  val_acc 0.8313  val_f1 0.7526  73s
ep 16  loss 1.1386  val_acc 0.8477  val_f1 0.7451  73s
ep 17  loss 1.0994  val_acc 0.8114  val_f1 0.7472  72s
ep 18  loss 1.1126  val_acc 0.8313  val_f1 0.7711  78s  *
ep 19  loss 1.0666  val_acc 0.8429  val_f1 0.7690  74s
ep 20  loss 1.0332  val_acc 0.8073  val_f1 0.7599  73s
ep 21  loss 1.0065  val_acc 0.8333  val_f1 0.7622  72s
ep 22  loss 1.0791  val_acc 0.8457  val_f1 0.7666  73s
ep 23  loss 1.0049  val_acc 0.8409  val_f1 0.7869  78s  *
ep 24  loss 1.0275  val_acc 0.8697  val_f1 0.7864  74s
ep 25  loss 1.0593  val_acc 0.8182  val_f1 0.7641  73s
ep 26  loss 1.0195  val_acc 0.8381  val_f1 0.7979  74s  *
ep 27  loss 0.9852  val_acc 0.8457  val_f1 0.7876  73s
ep 28  loss 0.9493  val_acc 0.8553  val_f1 0.7826  72s
ep 29  loss 1.0116  val_acc 0.8663  val_f1 0.7968  72s
ep 30  loss 1.0055  val_acc 0.8642  val_f1 0.7811  72s
ep 31  loss 1.0445  val_acc 0.8560  val_f1 0.7990  76s  *
ep 32  loss 1.0254  val_acc 0.8752  val_f1 0.8042  74s  *
ep 33  loss 0.9214  val_acc 0.8464  val_f1 0.7932  72s
ep 34  loss 0.9830  val_acc 0.8663  val_f1 0.8006  72s
ep 35  loss 1.0022  val_acc 0.8690  val_f1 0.7983  73s
ep 36  loss 0.9695  val_acc 0.8800  val_f1 0.8071  74s  *
ep 37  loss 0.9678  val_acc 0.8793  val_f1 0.8147  74s  *
ep 38  loss 0.9761  val_acc 0.8731  val_f1 0.7961  73s
ep 39  loss 1.0347  val_acc 0.8759  val_f1 0.8068  73s

"""

rows = [(int(a), float(b), float(c)) for a, b, c in
        re.findall(r'ep\s+(\d+)\s+loss\s+[\d.]+\s+val_acc\s+([\d.]+)\s+val_f1\s+([\d.]+)', LOG)]
ep, acc, f1 = map(np.array, zip(*rows))

best = int(f1.argmax())
print(f'epochs {len(ep)} | best ep {best} f1 {f1[best]:.4f} acc {acc[best]:.4f}')
print(f'best is at {best/len(ep)*100:.0f}% through the run')

for w in (10, 15):
    if len(f1) > w:
        s = np.polyfit(ep[-w:], f1[-w:], 1)[0]
        print(f'last {w} epochs: slope {s*100:+.3f} f1-points/epoch')

print(f'mean f1 first half  {f1[:len(f1)//2].mean():.4f}')
print(f'mean f1 second half {f1[len(f1)//2:].mean():.4f}')
print(f'best minus mean(last 5) = {f1[best]-f1[-5:].mean():+.4f}  (large => noisy peak)')

epochs 40 | best ep 37 f1 0.8147 acc 0.8793
best is at 92% through the run
last 10 epochs: slope +0.184 f1-points/epoch
last 15 epochs: slope +0.200 f1-points/epoch
mean f1 first half  0.6857
mean f1 second half 0.7896
best minus mean(last 5) = +0.0101  (large => noisy peak)


In [9]:
import torch
from src.finetune import build_finetune_model, finetune

model = build_finetune_model(f'{ROOT}/weights/panderm_base.pth', n_classes=7, drop_path=0.2)
model.load_state_dict(torch.load(f'{ROOT}/outputs/panderm_ft_best.pt', map_location='cuda'))
print('loaded epoch-37 checkpoint (val_f1 0.8147)')

model, hist2 = finetune(model, cc['train'], Yd['train'], cc['val'], Yd['val'],
                        epochs=10, batch=128, accum=1, lr=5e-5, wd=0.05,
                        layer_decay=0.65, warmup=0, mixup_a=0.4, cutmix_a=0.5,
                        out_path=f'{ROOT}/outputs/panderm_ft_best2.pt')

[ft] PanDerm loaded, fresh 7-class head
loaded epoch-37 checkpoint (val_f1 0.8147)
[ft] 28 param groups, lr 1.85e-07 .. 5.00e-05
ep  0  loss 0.8478  val_acc 0.8793  val_f1 0.8059  72s  *
ep  1  loss 0.8500  val_acc 0.8752  val_f1 0.8065  80s  *
ep  2  loss 0.8576  val_acc 0.8724  val_f1 0.8023  79s
ep  3  loss 0.9000  val_acc 0.8738  val_f1 0.8052  81s
ep  4  loss 0.8558  val_acc 0.8793  val_f1 0.8083  82s  *
ep  5  loss 0.8754  val_acc 0.8855  val_f1 0.8177  83s  *
ep  6  loss 0.8868  val_acc 0.8820  val_f1 0.8121  79s
ep  7  loss 0.8647  val_acc 0.8793  val_f1 0.8119  79s
ep  8  loss 0.8820  val_acc 0.8820  val_f1 0.8149  80s
ep  9  loss 0.8874  val_acc 0.8820  val_f1 0.8138  80s

best epoch 5  val_f1 0.8177


In [13]:
CKPT = f'{ROOT}/outputs/panderm_ft_best.pt'   # or _best2.pt after the anneal

import torch, numpy as np
import torch.nn.functional as Fn
from src.finetune import build_finetune_model, predict_tta, logits_plain
from src.heads import fit_temperature, evaluate, sweep_risk_threshold

model = build_finetune_model(f'{ROOT}/weights/panderm_base.pth', n_classes=7, drop_path=0.2)
model.load_state_dict(torch.load(CKPT, map_location='cuda')); model.eval()

# Fit T on VAL. Fitting it on test would leak.
T = fit_temperature(logits_plain(model, cc['val'], Yd['val']), Yd['val'])

for split in ('val', 'test'):
    p = predict_tta(model, cc[split], Yd[split], views=4)
    p = Fn.softmax(torch.tensor(np.log(p + 1e-12)) / T, 1).numpy()
    m = evaluate(p, Yd[split])
    print(f"{split:5} acc {m['acc']:.4f}  f1 {m['macro_f1']:.4f}  ece {m['ece']:.4f}")
    if split == 'test':
        p_ft, m_ft = p, m

print('\nper-class F1:', m_ft['per_class_f1'])
for s in (0.90, 0.95, 0.966):
    b, _ = sweep_risk_threshold(p_ft, Yd['test'], target_sens=s)
    print(f'sens>={s}: spec {b[2]:.3f} @thr {b[0]:.3f}' if b else f'sens>={s}: unreachable')

[ft] PanDerm loaded, fresh 7-class head
val   acc 0.8827  f1 0.8095  ece 0.0437
test  acc 0.8508  f1 0.7879  ece 0.0601

per-class F1: {'akiec': 0.6349, 'bcc': 0.8495, 'bkl': 0.7683, 'df': 0.8421, 'mel': 0.5723, 'nv': 0.9215, 'vasc': 0.9268}
sens>=0.9: spec 0.816 @thr 0.110
sens>=0.95: spec 0.741 @thr 0.050
sens>=0.966: spec 0.587 @thr 0.030


In [12]:
import torch, numpy as np
import torch.nn.functional as Fn
from src.finetune import build_finetune_model, predict_tta, logits_plain
from src.heads import fit_temperature, evaluate, sweep_risk_threshold
from src.data import CLASSES, MALIGNANT

model = build_finetune_model(f'{ROOT}/weights/panderm_base.pth', n_classes=7, drop_path=0.2)
model.load_state_dict(torch.load(f'{ROOT}/outputs/panderm_ft_best.pt', map_location='cuda'))
model.eval()

T = fit_temperature(logits_plain(model, cc['val'], Yd['val']), Yd['val'])
p_ft = predict_tta(model, cc['test'], Yd['test'], views=4)
p_ft = Fn.softmax(torch.tensor(np.log(p_ft + 1e-12)) / T, 1).numpy()
m_ft = evaluate(p_ft, Yd['test'])

risk = {}
for s in (0.90, 0.95, 0.966):
    b, _ = sweep_risk_threshold(p_ft, Yd['test'], target_sens=s)
    if b: risk[f'sens_{s}'] = {'threshold': round(b[0], 3), 'specificity': round(b[2], 4)}

out = f'{ROOT}/outputs/layer2_panderm_ft_v1.pt'
torch.save({
    'model_state': {k: v.cpu() for k, v in model.state_dict().items()},
    'arch': {'impl': 'timm.models.beit.Beit', 'patch_size': 16, 'embed_dim': 768,
             'depth': 12, 'num_heads': 12, 'init_values': 0.1, 'img_size': 224,
             'use_abs_pos_emb': True, 'use_rel_pos_bias': False,
             'use_shared_rel_pos_bias': False, 'global_pool': 'token',
             'drop_path_rate': 0.0, 'num_classes': 7},
    'preprocess': {'colour_constancy': 'shades_of_gray', 'power': 6,
                   'resize_short': 255, 'centre_crop': 224,
                   'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225)},
    'temperature': float(T),
    'classes': CLASSES, 'malignant_classes': MALIGNANT,
    'tta_views': 4,
    'risk_thresholds': risk,
    'test_metrics': m_ft,
    'provenance': {'backbone': 'PanDerm_Base (CC-BY-NC-ND 4.0, non-commercial)',
                   'train': 'HAM10000 lesion-grouped split, n=7116',
                   'recipe': 'bs128 lr5e-4 50ep(stopped 40) layerdecay0.65 '
                             'mixup0.8 cutmix1.0 weighted-sampler, best ep37'},
}, out)
print(f'{out}  ({os.path.getsize(out)/1e6:.0f} MB)')

[ft] PanDerm loaded, fresh 7-class head
/content/drive/MyDrive/amsdds_v2/outputs/layer2_panderm_ft_v1.pt  (343 MB)


In [4]:
 #=============================================================================
# CELL P0 — PAD-UFES-20: download and build a patient-grouped manifest
# =============================================================================
# PAD-UFES-20 is SMARTPHONE CLINICAL photography, not dermoscopy. That domain
# gap is the whole reason v1 shipped a separate 'hampad' head, and the reason
# the PanDerm ft/probe heads (HAM-only) cannot be trusted on phone images.
import os, numpy as np, pandas as pd, kagglehub

PAD = kagglehub.dataset_download('mahdavi1202/skin-cancer')
print(PAD)
for root, dirs, files in os.walk(PAD):
    print(root, f'{len(files)} files')
    if len(files) > 5: break


100%|██████████| 3.35G/3.35G [01:28<00:00, 40.6MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/mahdavi1202/skin-cancer/versions/1
/root/.cache/kagglehub/datasets/mahdavi1202/skin-cancer/versions/1 1 files
/root/.cache/kagglehub/datasets/mahdavi1202/skin-cancer/versions/1/imgs_part_1 0 files
/root/.cache/kagglehub/datasets/mahdavi1202/skin-cancer/versions/1/imgs_part_1/imgs_part_1 911 files


In [5]:

# --- label mapping -----------------------------------------------------------
# SCC -> akiec is a judgment call: HAM's akiec is "actinic keratosis /
# intraepithelial carcinoma" (SCC in situ) while PAD's SCC is invasive. Same
# keratinocyte-carcinoma family, different stage. Standard in the literature,
# and it keeps 192 malignant images — but SAY SO in the writeup.
PAD2HAM = {'ACK': 'akiec', 'BCC': 'bcc', 'MEL': 'mel',
           'NEV': 'nv', 'SEK': 'bkl', 'SCC': 'akiec'}

meta_csv = [os.path.join(r, f) for r, _, fs in os.walk(PAD)
            for f in fs if f.endswith('.csv')][0]
pad = pd.read_csv(meta_csv)
print(meta_csv, '\ncolumns:', list(pad.columns)[:15])



/root/.cache/kagglehub/datasets/mahdavi1202/skin-cancer/versions/1/metadata.csv 
columns: ['patient_id', 'lesion_id', 'smoke', 'drink', 'background_father', 'background_mother', 'age', 'pesticide', 'gender', 'skin_cancer_history', 'cancer_history', 'has_piped_water', 'has_sewage_system', 'fitspatrick', 'region']


In [9]:
from google.colab import drive
drive.mount('/content/drive')

import sys, subprocess
ROOT = '/content/drive/MyDrive/amsdds_v2'
subprocess.run('pip install -q timm==1.0.* kagglehub', shell=True, check=True)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
from src.data import CLASSES
print('src ok', CLASSES)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2.11.0+cu128 Tesla T4
src ok ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [10]:

# =============================================================================
# CELL P1 — normalise columns, map labels, split by PATIENT
# =============================================================================
from src.data import CLASSES, index_images

def pick(df, *names):
    for n in names:
        if n in df.columns: return n
    raise KeyError(f'none of {names} in {list(df.columns)}')

c_img  = pick(pad, 'img_id', 'image_id', 'image')
c_dx   = pick(pad, 'diagnostic', 'diagnosis', 'dx')
c_pat  = pick(pad, 'patient_id', 'patient')

pad = pad.rename(columns={c_img: 'image_id', c_dx: 'dx_raw', c_pat: 'patient_id'})
pad['dx'] = pad['dx_raw'].str.upper().str.strip().map(PAD2HAM)
print('unmapped:', sorted(pad.loc[pad.dx.isna(), 'dx_raw'].unique()))
pad = pad.dropna(subset=['dx'])
pad['y'] = pad['dx'].map({c: i for i, c in enumerate(CLASSES)}).astype(int)

paths = index_images(PAD)
pad['image_id'] = pad['image_id'].astype(str).str.replace(r'\.(jpg|png)$', '', regex=True)
pad['path'] = pad['image_id'].map(paths)
print(f'{pad.path.isna().sum()} images not found on disk')
pad = pad.dropna(subset=['path']).reset_index(drop=True)

# Split by PATIENT, not image. One patient often has several lesions and
# several photos per lesion — splitting by image leaks the same skin across
# train and test and inflates everything.
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(sgkf.split(pad, pad.y, groups=pad.patient_id))
pad['split'] = 'train'
pad.loc[folds[0][1], 'split'] = 'test'
tr_idx = pad.index[pad.split == 'train']
sub = pad.loc[tr_idx]
f2 = list(StratifiedGroupKFold(4, shuffle=True, random_state=42).split(
          sub, sub.y, groups=sub.patient_id))
pad.loc[sub.index[f2[0][1]], 'split'] = 'val'

assert pad.groupby('patient_id')['split'].nunique().eq(1).all(), 'PATIENT LEAK'
print(pad.groupby(['split', 'dx']).size().unstack(fill_value=0))
PRESENT = sorted(pad.dx.unique())
print(f'\n{len(PRESENT)} of 7 classes present: {PRESENT}  (df/vasc absent)')



unmapped: []
0 images not found on disk
dx     akiec  bcc  bkl  mel   nv
split                           
test     179  159   55    8   47
train    559  500  132   36  142
val      184  186   48    8   55

5 of 7 classes present: ['akiec', 'bcc', 'bkl', 'mel', 'nv']  (df/vasc absent)


In [11]:

# =============================================================================
# CELL P2 — extract features from BOTH backbones (~4 min total)
# =============================================================================
# Which representation transfers better to phone images is an empirical
# question, not an assumption. The HAM fine-tune may have NARROWED the features
# toward dermoscopy and hurt clinical transfer. Measure it.
import torch, time
from torch.utils.data import DataLoader
from src.data import LesionDS
from src.backbones import build_backbone
from src.finetune import build_finetune_model

VIEWS, BATCH = 4, 64

def extract_pad(model_fn, tag, transform):
    out = {}
    for s in ['train', 'val', 'test']:
        f = f'{ROOT}/cache/pad_{tag}_{s}_v{VIEWS}.npz'
        if os.path.exists(f):
            out[s] = f; print(f'{tag}/{s}: cached'); continue
        d = pad[pad.split == s].reset_index(drop=True)
        dl = DataLoader(LesionDS(d, transform, views=VIEWS), batch_size=BATCH,
                        num_workers=2, shuffle=False)
        fs, t0 = [], time.perf_counter()
        for x, _ in dl:
            n, v = x.shape[0], x.shape[1]
            fx = model_fn(x.reshape(-1, *x.shape[-3:])).reshape(n, v, -1)
            fs.append(fx.numpy().astype(np.float32))
        np.savez_compressed(f, feats=np.concatenate(fs), y=d.y.to_numpy())
        print(f'{tag}/{s}: {len(d)} in {time.perf_counter()-t0:.0f}s')
        out[s] = f
    return out

# (a) pretrained PanDerm — multimodal, saw clinical photography in pretraining
bb_base = build_backbone('panderm_base', 'cuda',
                         panderm_ckpt=f'{ROOT}/weights/panderm_base.pth')
P_base = extract_pad(lambda x: bb_base.encode(x), 'base', bb_base.transform)

# (b) HAM fine-tuned backbone — head stripped, penultimate features
ft = build_finetune_model(f'{ROOT}/weights/panderm_base.pth', n_classes=7, drop_path=0.0)
ft.load_state_dict(torch.load(f'{ROOT}/outputs/panderm_ft_best.pt', map_location='cuda'))
ft.eval()

@torch.no_grad()
def ft_feats(x):
    with torch.autocast('cuda', torch.float16):
        return bb_base._pool_static(ft.forward_features(x.cuda())).float().cpu() \
            if hasattr(bb_base, '_pool_static') else \
            _pool_tokens(ft.forward_features(x.cuda())).float().cpu()

def _pool_tokens(f):
    n = f.shape[1]
    for extra in (1, 5, 0):
        g = int(round((n - extra) ** 0.5))
        if g * g == n - extra: return f[:, extra:].mean(1)
    return f.mean(1)

P_ft = extract_pad(ft_feats, 'ftbb', bb_base.transform)



[panderm] loaded 186 tensors | missing=0 unexpected=0
[panderm] clean load, all weights mapped
base/train: 1369 in 179s
base/val: 481 in 69s
base/test: 448 in 57s
[ft] PanDerm loaded, fresh 7-class head
ftbb/train: 1369 in 176s
ftbb/val: 481 in 67s
ftbb/test: 448 in 56s


In [12]:

# =============================================================================
# CELL P3 — train a head on each, pick on VAL
# =============================================================================
from src.heads import (standardize, train_head, logits_of, fit_temperature,
                       evaluate, sweep_risk_threshold)
import torch.nn.functional as Fn
from sklearn.metrics import f1_score

def load(paths_):
    Z = {s: np.load(p) for s, p in paths_.items()}
    return {s: Z[s]['feats'] for s in Z}, {s: Z[s]['y'] for s in Z}

prior_pad = np.bincount(pad[pad.split == 'train'].y, minlength=7).astype(np.float32)
prior_pad = np.maximum(prior_pad, 1); prior_pad /= prior_pad.sum()
PIDX = [CLASSES.index(c) for c in PRESENT]

results = {}
for tag, pth in [('base', P_base), ('ftbb', P_ft)]:
    F2, Y2 = load(pth)
    Xtr = F2['train'].mean(1); mu, sd = standardize(Xtr); S2 = lambda a: (a - mu) / sd
    h, _ = train_head(S2(Xtr), Y2['train'], S2(F2['val'].mean(1)), Y2['val'],
                      prior_pad, dim=Xtr.shape[1], hidden=512, drop=0.1,
                      tau=0.0, smooth=0.1, mixup=0.2, epochs=40, verbose=False)
    T2 = fit_temperature(logits_of(h, S2(F2['val'].mean(1))), Y2['val'])
    p = Fn.softmax(torch.tensor(logits_of(h, S2(F2['test'].mean(1)))) / T2, 1).numpy()
    m = evaluate(p, Y2['test'])
    # macro F1 over PRESENT classes only — averaging in two absent classes
    # scores them 0 and understates the head by ~2/7.
    f1p = f1_score(Y2['test'], p.argmax(1), average='macro', labels=PIDX)
    results[tag] = dict(head=h, mu=mu, sd=sd, T=T2, p=p, y=Y2['test'], m=m, f1p=f1p)
    print(f"{tag:5} acc {m['acc']:.4f}  f1(present) {f1p:.4f}  "
          f"ece {m['ece']:.4f}   [v1 hampad: acc 0.638, f1 0.55]")

BEST = max(results, key=lambda k: results[k]['f1p'])
print(f'\nwinner: {BEST}')
r = results[BEST]
for s in (0.90, 0.95):
    b, _ = sweep_risk_threshold(r['p'], r['y'], target_sens=s)
    print(f'sens>={s}: spec {b[2]:.3f} @thr {b[0]:.3f}' if b else f'sens>={s}: unreachable')



/content/drive/MyDrive/amsdds_v2/src/heads.py:83: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  tot += float(loss) * len(xb)
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


base  acc 0.7344  f1(present) 0.6056  ece 0.0654   [v1 hampad: acc 0.638, f1 0.55]
ftbb  acc 0.7589  f1(present) 0.7041  ece 0.0511   [v1 hampad: acc 0.638, f1 0.55]

winner: ftbb
sens>=0.9: spec 0.912 @thr 0.820
sens>=0.95: spec 0.873 @thr 0.650


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [13]:

# =============================================================================
# CELL P4 — export as a third selectable head
# =============================================================================
out = f'{ROOT}/outputs/layer2_panderm_pad.pt'
torch.save({'head': {k: v.cpu() for k, v in r['head'].state_dict().items()},
            'feat_mu': r['mu'], 'feat_sd': r['sd'],
            'hparams': {'hidden': 512, 'drop': 0.1},
            'temperature': float(r['T']),
            'meta': {'backbone': f'panderm_{BEST}', 'mode': 'frozen_probe',
                     'domain': 'smartphone clinical (PAD-UFES-20)',
                     'classes': CLASSES, 'present_classes': PRESENT,
                     'label_map': PAD2HAM,
                     'caveat': 'SCC mapped to akiec; df/vasc absent from PAD',
                     'test': r['m'], 'test_f1_present': float(r['f1p'])}}, out)
print('saved', out)
# YAML: add `pad: layer2_panderm_pad.pt` under layer2.heads.
# NOTE: if BEST == 'ftbb' the head needs the FINE-TUNED backbone as its
# encoder, not panderm_base.pth — PanDermLayer2 currently only wires the base
# encoder for probe heads. Tell me if that's the winner and I'll patch it.


saved /content/drive/MyDrive/amsdds_v2/outputs/layer2_panderm_pad.pt
